In [11]:
import re
import time
import random
import requests
import pandas as pd

from pathlib import Path
from bs4 import BeautifulSoup

ROOT = Path.cwd().parent
print(ROOT)

c:\Users\sebas\PycharmProjects\Git\Indiana-Jones-and-the-Domestic-Box-Office-Forecasting-Model


In [46]:
DATA_PROCESSED = ROOT/"data/processed"
CACHE_DIR = ROOT/"data/cache/the_numbers_pages"

CACHE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_PROCESSED / "scrape_ready_us_production_filtered.csv"
OUTPUT_PATH = DATA_PROCESSED / "the_numbers_scraped_movie_details.csv"

scrape_df = pd.read_csv(INPUT_PATH)

print(scrape_df.shape)
print(scrape_df.columns.tolist())

scrape_df = scrape_df[
    scrape_df["the_numbers_url"].notna()
].copy()

scrape_df = scrape_df.drop_duplicates(subset="tconst").reset_index(drop=True)

print(scrape_df.shape)
scrape_df.head()

(2316, 9)
['tconst', 'primaryTitle', 'startYear', 'the_numbers_url', 'found_url', 'normalized_title', 'kaggle_year', 'kaggle_title', 'Production_Countries']
(2316, 9)


,tconst,primaryTitle,startYear,the_numbers_url,found_url,normalized_title,kaggle_year,kaggle_title,Production_Countries
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,fantastic four,2005,Fantastic Four,"Germany, United States of America"
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,corpse bride,2005,Corpse Bride,"United Kingdom, United States of America"
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,star wars episode iii revenge of the sith,2005,Star Wars: Episode III - Revenge of the Sith,United States of America
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,hellboy,2004,Hellboy,"Czech Republic, United States of America"
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,the bank job,2008,The Bank Job,"Australia, United Kingdom, United States of Am..."


In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

SLEEP_SECONDS = 2.5

In [14]:
def clean_cache_name(text):
    text = str(text)
    text = re.sub(r"[^a-zA-Z0-9]+", "_", text)
    return text.strip("_")[:120]


def fetch_the_numbers_html(row):
    tconst = row["tconst"]
    url = row["the_numbers_url"]

    cache_path = CACHE_DIR / f"{clean_cache_name(tconst)}.html"

    if cache_path.exists():
        return cache_path.read_text(encoding="utf-8", errors="ignore")

    response = requests.get(
        url,
        headers=headers,
        timeout=25
    )

    response.raise_for_status()

    html = response.text

    cache_path.write_text(html, encoding="utf-8", errors="ignore")

    time.sleep(SLEEP_SECONDS + random.uniform(0, 1))

    return html

In [15]:
def money_to_int(text):
    if not text or pd.isna(text):
        return None

    text = str(text)

    match = re.search(r"\$([0-9,]+)", text)

    if not match:
        return None

    return int(match.group(1).replace(",", ""))


def number_to_int(text):
    if not text or pd.isna(text):
        return None

    text = str(text)

    match = re.search(r"([0-9,]+)", text)

    if not match:
        return None

    return int(match.group(1).replace(",", ""))


def parse_float(text):
    if not text or pd.isna(text):
        return None

    match = re.search(r"([0-9]+(?:\.[0-9]+)?)", str(text))

    if not match:
        return None

    return float(match.group(1))


def clean_ordinal_date(text):
    return re.sub(r"(\d+)(st|nd|rd|th)", r"\1", str(text))


def parse_first_date(text):
    if not text or pd.isna(text):
        return None

    date_pattern = (
        r"(January|February|March|April|May|June|July|August|September|"
        r"October|November|December)\s+\d{1,2}(?:st|nd|rd|th)?,\s+\d{4}"
    )

    match = re.search(date_pattern, str(text))

    if not match:
        return None

    date_text = clean_ordinal_date(match.group(0))

    return pd.to_datetime(date_text, errors="coerce").date()

In [ ]:
LABELS = [
    "Opening Weekend",
    "Legs",
    "Domestic Share",
    "Production Budget",
    "Theater counts",
    "Infl. Adj. Dom. BO",
    "Domestic Releases",
    "International Releases",
    "Video Release",
    "MPA Rating",
    "Running Time",
    "Cast, crew, or production detail",
    "Plot point",
    "Social setting",
    "Subgenre",
    "Source",
    "Genre",
    "Production Method",
    "Creative Type",
    "Production/Financing Companies",
    "Production Companies",
    "Financing Companies",
    "Production Countries",
    "Languages",
    "Box Office",
    "Domestic Box Office",
    "International Box Office",
    "Worldwide Box Office",
    "Target audience",
    "Time period setting",
]

SECTION_HEADERS = {
    "Metrics",
    "Movie Details",
    "Box Office",
    "Financial Analysis",
    "Ranking on other Records and Milestones",
    "Weekend Box Office Performance",
    "Daily Box Office Performance",
    "Weekly Box Office Performance",
    "International Box Office Performance",
}

def cut_off_embedded_labels(text, current_label=None):
    if not text:
        return None

    text = str(text).strip()

    for label in LABELS:
        if label == current_label:
            continue

        # catches: " Faith-Based Drama Target audience: Faith-Based"
        pattern = rf"\s+{re.escape(label)}:"
        match = re.search(pattern, text)

        if match:
            text = text[:match.start()].strip()

    return text if text else None

def get_lines_from_html(html):
    soup = BeautifulSoup(html, "html.parser")

    lines = [
        re.sub(r"\s+", " ", x).strip()
        for x in soup.stripped_strings
    ]

    return [line for line in lines if line]


def is_known_label(line):
    line = line.strip()

    for label in LABELS:
        if line == f"{label}:":
            return True

        if line.startswith(f"{label}:"):
            return True

        # catches cases like "Box Office" without colon
        if line == label:
            return True

    return False


def get_field(lines, label):
    for i, line in enumerate(lines):
        if line == f"{label}:" or line.startswith(f"{label}:"):
            value = ""

            if ":" in line:
                value = line.split(":", 1)[1].strip()

            parts = []

            if value:
                parts.append(value)

            j = i + 1

            while j < len(lines):
                next_line = lines[j].strip()

                if next_line in SECTION_HEADERS:
                    break

                if is_known_label(next_line):
                    break

                parts.append(next_line)
                j += 1

            joined = " ".join(parts).strip()

            joined = cut_off_embedded_labels(joined, current_label=label)

            return joined if joined else None

    return None

In [24]:
def parse_mpa_rating(text):
    if not text:
        return None, None

    text = str(text).strip()

    match = re.match(r"^(PG-13|NC-17|PG|G|R)\b\s*(.*)", text)

    if not match:
        return None, text

    rating = match.group(1)
    reason = match.group(2).strip()

    # clean leading filler
    reason = re.sub(r"^for\s+", "", reason, flags=re.IGNORECASE)

    return rating, reason

In [30]:
def clean_distributor(distributor):
    if not distributor:
        return None

    distributor = str(distributor).strip()

    # Stop if another date starts after the distributor name
    distributor = re.split(
        r"\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2}",
        distributor
    )[0].strip()

    # Stop if another release type starts
    distributor = re.split(
        r"\s+\((Wide|Limited|Expands Wide)\)",
        distributor
    )[0].strip()

    # Clean trailing punctuation
    distributor = distributor.strip(" ,;")

    return distributor if distributor else None

In [32]:
def parse_domestic_release_info(domestic_releases_text):
    if not domestic_releases_text:
        return {
            "domestic_release_date": None,
            "release_type": None,
            "distributor": None,
            "all_domestic_release_types": None,
        }

    text = str(domestic_releases_text)

    release_date = parse_first_date(text)

    valid_types = {"Limited", "Wide", "Expands Wide"}

    release_types = [
        x for x in re.findall(r"\(([^)]+)\)", text)
        if x in valid_types
    ]

    release_types = list(dict.fromkeys(release_types))

    release_type = release_types[0] if release_types else None

    distributor = None

    dist_match = re.search(r"\)\s+by\s+(.+)", text)

    if not dist_match:
        dist_match = re.search(r"\bby\s+(.+)", text)

    if dist_match:
        distributor = clean_distributor(dist_match.group(1))

    return {
        "domestic_release_date": release_date,
        "release_type": release_type,
        "distributor": distributor,
        "all_domestic_release_types": "; ".join(release_types) if release_types else None,
    }

In [38]:
def parse_the_numbers_page(html):
    lines = get_lines_from_html(html)

    opening_weekend_text = get_field(lines, "Opening Weekend")
    theater_counts_text = get_field(lines, "Theater counts")
    domestic_releases_text = get_field(lines, "Domestic Releases")
    production_budget_text = get_field(lines, "Production Budget")
    legs_text = get_field(lines, "Legs")
    runtime_text = get_field(lines, "Running Time")

    mpa_text = get_field(lines, "MPA Rating")
    mpa_rating_code, mpa_rating_reason = parse_mpa_rating(mpa_text)

    release_info = parse_domestic_release_info(domestic_releases_text)

    opening_theaters = None

    if theater_counts_text:
        match = re.search(r"([0-9,]+)\s+opening theaters", theater_counts_text)
        if match:
            opening_theaters = int(match.group(1).replace(",", ""))

    runtime_minutes = None

    if runtime_text:
        match = re.search(r"([0-9]+)\s+minutes", runtime_text)
        if match:
            runtime_minutes = int(match.group(1))

    parsed = {
        "opening_weekend_gross": money_to_int(opening_weekend_text),
        "opening_theaters": opening_theaters,
        "domestic_release_date": release_info["domestic_release_date"],
        "release_type": release_info["release_type"],
        "all_domestic_release_types": release_info["all_domestic_release_types"],
        "distributor": release_info["distributor"],
        "production_budget": money_to_int(production_budget_text),
        "MPA_rating": mpa_rating_code,
        "MPA_rating_reason": mpa_rating_reason,
        "runtime_minutes": runtime_minutes,
        "genre": get_field(lines, "Genre"),
        "subgenre": get_field(lines, "Subgenre"),
        "target_audience": get_field(lines, "Target audience"),
        "time_period_setting": get_field(lines, "Time period setting"),
        "source": get_field(lines, "Source"),
        "production_method": get_field(lines, "Production Method"),
        "creative_type": get_field(lines, "Creative Type"),
        "production_countries": get_field(lines, "Production Countries"),
        "languages": get_field(lines, "Languages"),
        "legs": parse_float(legs_text),
        "plot_point": get_field(lines, "Plot point"),

        # keep raw text for debugging
        "raw_opening_weekend_text": opening_weekend_text,
        "raw_theater_counts_text": theater_counts_text,
        "raw_domestic_releases_text": domestic_releases_text,
    }

    return parsed

TESTING 1 FILM

In [39]:
test_row = scrape_df.iloc[0]

html = fetch_the_numbers_html(test_row)

test_parsed = parse_the_numbers_page(html)

test_parsed

{'opening_weekend_gross': 56061504,
 'opening_theaters': 3602,
 'domestic_release_date': datetime.date(2005, 7, 8),
 'release_type': 'Wide',
 'all_domestic_release_types': 'Wide',
 'distributor': '20th Century Fox',
 'production_budget': 87500000,
 'MPA_rating': 'PG-13',
 'MPA_rating_reason': 'sequences of intense action, and some suggestive content.',
 'runtime_minutes': 106,
 'genre': 'Action',
 'subgenre': 'Action Adventure',
 'target_audience': None,
 'time_period_setting': None,
 'source': 'Based on Comic/Graphic Novel',
 'production_method': 'Live Action',
 'creative_type': 'Super Hero',
 'production_countries': 'United States',
 'languages': 'English',
 'legs': 2.76,
 'plot_point': 'Friends turned Enemies, Origin Story, Revenge',
 'raw_opening_weekend_text': '$56,061,504 (36.2% of total gross)',
 'raw_theater_counts_text': '3,602 opening theaters/3,619 max. theaters, 5.9 weeks average run per theater',
 'raw_domestic_releases_text': 'July 8th, 2005 (Wide) by 20th Century Fox'}

TESTING 20 FILMS

In [40]:
sample_scrape_df = scrape_df.sample(20, random_state=42).copy()

sample_rows = []

for _, row in sample_scrape_df.iterrows():
    record = {
        "tconst": row["tconst"],
        "primaryTitle": row["primaryTitle"],
        "startYear": row["startYear"],
        "the_numbers_url": row["the_numbers_url"],
    }

    try:
        html = fetch_the_numbers_html(row)
        parsed = parse_the_numbers_page(html)
        record.update(parsed)
        record["scrape_success"] = True
        record["scrape_error"] = None

    except Exception as e:
        record["scrape_success"] = False
        record["scrape_error"] = str(e)

    sample_rows.append(record)

sample_result_df = pd.DataFrame(sample_rows)

sample_result_df[[
    "primaryTitle",
    "opening_weekend_gross",
    "opening_theaters",
    "domestic_release_date",
    "release_type",
    "distributor",
    "production_budget",
    "MPA_rating",
    "runtime_minutes",
    "genre",
    "subgenre",
    "target_audience",
    "time_period_setting",
    "source",
    "scrape_success"
]]

,primaryTitle,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,distributor,production_budget,MPA_rating,runtime_minutes,genre,subgenre,target_audience,time_period_setting,source,scrape_success
0,Breakthrough,11282333,2824,2019-04-17,Wide,20th Century Fox,14000000.0,PG,116,Drama,Faith-Based Drama,Faith-Based Film,NaN,Based on Factual Book/Article,True
1,The Covenant,8852458,2681,2006-09-08,Wide,Sony Pictures,20000000.0,PG-13,97,Horror,Action Horror,NaN,NaN,Based on Comic/Graphic Novel,True
2,Miracle,19377577,2605,2004-02-06,Wide,Walt Disney,28000000.0,PG,135,Drama,Sports Drama,NaN,NaN,Based on Real Life Events,True
3,Bones and All,121004,5,2022-11-18,Limited,United Artists,NaN,R,130,Thriller/Suspense,Psychological Thriller,NaN,1980s,Based on Fiction Book/Short Story,True
4,Seed of Chucky,8774520,2059,2004-11-12,Wide,Focus/Rogue Pictures,29000000.0,R,87,Horror,Horror Comedy,NaN,NaN,Original Screenplay,True
5,She's the Man,10730372,2623,2006-03-17,Wide,Paramount Pictures,25000000.0,PG-13,105,Romantic Comedy,Romantic Comedy,NaN,NaN,Based on Play,True
6,The Menu,9004957,3211,2022-11-18,Wide,Searchlight Pictures,30000000.0,R,107,Black Comedy,Black Comedy,NaN,NaN,Original Screenplay,True
7,The Ministry of Ungentlemanly Warfare,8913698,2845,2024-04-19,Wide,Lionsgate,60000000.0,R,120,Action,Action Comedy,NaN,1930s,Based on Factual Book/Article,True
8,Spanglish,8817853,2438,2004-12-17,Wide,Sony Pictures,75000000.0,PG-13,133,Comedy,Comedy Drama,Hispanic,NaN,Original Screenplay,True
9,Transformers: Age of Extinction,100038390,4233,2014-06-27,Wide,Paramount Pictures,210000000.0,PG-13,165,Action,Action Adventure,NaN,NaN,Based on TV,True


In [35]:
sample_result_df.isna().mean().sort_values(ascending=False)

scrape_error                  1.0
production_budget             0.1
languages                     0.1
tconst                        0.0
the_numbers_url               0.0
opening_weekend_gross         0.0
domestic_release_date         0.0
opening_theaters              0.0
release_type                  0.0
all_domestic_release_types    0.0
startYear                     0.0
primaryTitle                  0.0
MPA_rating                    0.0
distributor                   0.0
MPA_rating_reason             0.0
runtime_minutes               0.0
source                        0.0
production_method             0.0
genre                         0.0
subgenre                      0.0
production_countries          0.0
creative_type                 0.0
plot_point                    0.0
legs                          0.0
raw_opening_weekend_text      0.0
raw_theater_counts_text       0.0
raw_domestic_releases_text    0.0
scrape_success                0.0
dtype: float64

In [36]:
sample_result_df[[
    "primaryTitle",
    "scrape_success",
    "scrape_error",
    "production_budget",
    "languages"
]].value_counts(dropna=False)

primaryTitle                           scrape_success  scrape_error  production_budget  languages       
Breakthrough                           True            NaN           14000000.0         English             1
The Covenant                           True            NaN           20000000.0         English             1
Miracle                                True            NaN           28000000.0         English             1
Bones and All                          True            NaN           NaN                English             1
Seed of Chucky                         True            NaN           29000000.0         NaN                 1
She's the Man                          True            NaN           25000000.0         English             1
The Menu                               True            NaN           30000000.0         English             1
The Ministry of Ungentlemanly Warfare  True            NaN           60000000.0         English             1
Spanglish      

FULL SCRAPING

In [41]:
existing_df = None
done_tconsts = set()

if OUTPUT_PATH.exists():
    existing_df = pd.read_csv(OUTPUT_PATH)
    done_tconsts = set(existing_df["tconst"].dropna())
    print(f"Resuming. Already scraped: {len(done_tconsts)}")
else:
    print("Starting fresh.")

Starting fresh.


In [42]:
SAVE_EVERY = 50

new_rows = []

start_time = time.time()

todo_df = scrape_df[
    ~scrape_df["tconst"].isin(done_tconsts)
].copy()

print(f"Movies left to scrape: {len(todo_df)}")

for i, (_, row) in enumerate(todo_df.iterrows(), start=1):

    record = {
        "tconst": row["tconst"],
        "primaryTitle": row.get("primaryTitle"),
        "startYear": row.get("startYear"),
        "the_numbers_url": row["the_numbers_url"],
        "scrape_success": False,
        "scrape_error": None,
    }

    try:
        html = fetch_the_numbers_html(row)
        parsed = parse_the_numbers_page(html)

        record.update(parsed)
        record["scrape_success"] = True

    except Exception as e:
        record["scrape_error"] = str(e)

    new_rows.append(record)

    if i % SAVE_EVERY == 0:
        elapsed_min = (time.time() - start_time) / 60
        success_count = sum(r["scrape_success"] for r in new_rows)

        print(
            f"[{i}/{len(todo_df)}] "
            f"Success: {success_count} | "
            f"Elapsed: {elapsed_min:.2f} min"
        )

        temp_df = pd.DataFrame(new_rows)

        if existing_df is not None:
            combined_df = pd.concat([existing_df, temp_df], ignore_index=True)
        else:
            combined_df = temp_df

        combined_df = combined_df.drop_duplicates(
            subset="tconst",
            keep="last"
        )

        combined_df.to_csv(OUTPUT_PATH, index=False)

Movies left to scrape: 2316
[50/2316] Success: 50 | Elapsed: 2.83 min
[100/2316] Success: 100 | Elapsed: 5.71 min
[150/2316] Success: 150 | Elapsed: 8.57 min
[200/2316] Success: 200 | Elapsed: 11.43 min
[250/2316] Success: 250 | Elapsed: 14.29 min
[300/2316] Success: 300 | Elapsed: 17.08 min
[350/2316] Success: 350 | Elapsed: 19.99 min
[400/2316] Success: 400 | Elapsed: 22.91 min
[450/2316] Success: 450 | Elapsed: 25.76 min
[500/2316] Success: 500 | Elapsed: 28.61 min
[550/2316] Success: 550 | Elapsed: 31.51 min
[600/2316] Success: 600 | Elapsed: 34.52 min
[650/2316] Success: 650 | Elapsed: 37.56 min
[700/2316] Success: 700 | Elapsed: 40.42 min
[750/2316] Success: 750 | Elapsed: 43.36 min
[800/2316] Success: 800 | Elapsed: 46.30 min
[850/2316] Success: 850 | Elapsed: 49.21 min
[900/2316] Success: 900 | Elapsed: 52.20 min
[950/2316] Success: 950 | Elapsed: 55.08 min
[1000/2316] Success: 1000 | Elapsed: 58.06 min
[1050/2316] Success: 1050 | Elapsed: 60.93 min
[1100/2316] Success: 1100 | 

In [47]:
final_new_df = pd.DataFrame(new_rows)

if existing_df is not None:
    final_df = pd.concat([existing_df, final_new_df], ignore_index=True)
else:
    final_df = final_new_df

final_df = final_df.drop_duplicates(
    subset="tconst",
    keep="last"
)

final_df.to_csv(OUTPUT_PATH, index=False)

print(final_df.shape)
print(final_df["scrape_success"].value_counts(dropna=False))

final_df.head()

(2316, 30)
scrape_success
True    2316
Name: count, dtype: int64


,tconst,primaryTitle,startYear,the_numbers_url,scrape_success,scrape_error,opening_weekend_gross,opening_theaters,domestic_release_date,release_type,...,source,production_method,creative_type,production_countries,languages,legs,plot_point,raw_opening_weekend_text,raw_theater_counts_text,raw_domestic_releases_text
0,tt0120667,Fantastic Four,2005.0,https://www.the-numbers.com/movie/Fantastic-Fo...,True,None,56061504.0,3602.0,2005-07-08,Wide,...,Based on Comic/Graphic Novel,Live Action,Super Hero,United States,English,2.76,"Friends turned Enemies, Origin Story, Revenge","$56,061,504 (36.2% of total gross)","3,602 opening theaters/3,619 max. theaters, 5....","July 8th, 2005 (Wide) by 20th Century Fox"
1,tt0121164,Corpse Bride,2005.0,https://www.the-numbers.com/movie/Corpse-Bride,True,None,19145480.0,3204.0,2005-09-16,Expands Wide,...,Based on Folk Tale/Legend/Fairytale,Stop-Motion Animation,Fantasy,United States,English,2.85,"Arranged Marriage, Friendly Ghost, Romance, Zo...","$19,145,480 (35.1% of total gross)","3,204 opening theaters/3,204 max. theaters, 5....","September 16th, 2005 (Special Engagement) by W..."
2,tt0121766,Star Wars: Episode III - Revenge of the Sith,2005.0,https://www.the-numbers.com/movie/Star-Wars-Ep...,True,None,108435841.0,3661.0,2005-05-19,Wide,...,Original Screenplay,Animation/Live Action,Science Fiction,United States,English,3.82,"Betrayal, Cloning, Cyborg, Death of a Spouse o...","$108,435,841 (26.2% of total gross)","3,661 opening theaters/3,663 max. theaters, 8....","May 19th, 2005 (Wide) by 20th Century Fox Apri..."
3,tt0167190,Hellboy,2004.0,https://www.the-numbers.com/movie/Hellboy,True,None,23172440.0,3028.0,2004-04-02,Wide,...,Based on Comic/Graphic Novel,Live Action,Super Hero,United States,English,2.57,Demons Source material: Dark Horse Comics,"$23,172,440 (38.9% of total gross)","3,028 opening theaters/3,043 max. theaters, 4....","April 2nd, 2004 (Wide) by Sony Pictures"
4,tt0200465,The Bank Job,2008.0,https://www.the-numbers.com/movie/Bank-Job-The,True,None,5935256.0,1603.0,2008-03-07,Wide,...,Based on Real Life Events,Live Action,Dramatization,United Kingdom,English,5.06,"Bank Robbery, Blackmail, Corrupt Cops, Heist, ...","$5,935,256 (19.7% of total gross)","1,603 opening theaters/1,613 max. theaters, 6....","March 7th, 2008 (Wide) by Lionsgate"


In [48]:
target_cols = [
    "opening_weekend_gross",
    "opening_theaters",
    "domestic_release_date",
    "release_type",
    "distributor",
    "production_budget",
    "MPA_rating",
    "runtime_minutes",
    "genre",
    "subgenre",
    "target_audience",
    "time_period_setting",
    "source",
    "production_method",
    "creative_type",
    "production_countries",
    "languages",
    "legs",
    "plot_point",
]

missing_summary = final_df[target_cols].isna().mean().sort_values(ascending=False)

missing_summary

target_audience          0.953800
time_period_setting      0.823402
production_budget        0.087219
plot_point               0.056995
subgenre                 0.026339
opening_weekend_gross    0.026339
opening_theaters         0.026339
runtime_minutes          0.022453
legs                     0.021589
languages                0.021589
release_type             0.015544
domestic_release_date    0.013385
distributor              0.013385
MPA_rating               0.007772
production_countries     0.006477
genre                    0.002591
creative_type            0.002591
source                   0.002591
production_method        0.002591
dtype: float64

In [49]:
model_base = final_df.copy()

model_base = model_base[
    model_base["opening_weekend_gross"].notna()
].copy()

print(final_df.shape)
print(model_base.shape)

(2316, 30)
(2255, 30)


In [50]:
model_base.to_csv(
    ROOT/"data/processed/the_numbers_model_base_v1.csv",
    index=False
)

In [51]:
model_base_missing_summary = model_base[target_cols].isna().mean().sort_values(ascending=False)

model_base_missing_summary


target_audience          0.952993
time_period_setting      0.821729
production_budget        0.073614
plot_point               0.048780
languages                0.019956
runtime_minutes          0.015965
subgenre                 0.014191
production_countries     0.004878
MPA_rating               0.003991
release_type             0.002217
creative_type            0.001774
distributor              0.001330
genre                    0.000887
production_method        0.000887
source                   0.000887
domestic_release_date    0.000000
opening_theaters         0.000000
opening_weekend_gross    0.000000
legs                     0.000000
dtype: float64